In [59]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# -------------------------------------------------
# 1. Load dataset
# -------------------------------------------------
csv_path = "IMDb Movies India.csv"
df = pd.read_csv(csv_path, encoding="utf-8", encoding_errors="ignore")

TARGET_COL = "Rating"

# -------------------------------------------------
# 2. Clean numeric columns
# -------------------------------------------------
# Year from "(2019)"
df["Year"] = df["Year"].astype(str).str.extract(r"(\d{4})")
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

# Duration from "109 min"
df["Duration"] = df["Duration"].astype(str).str.replace("min", "", regex=False)
df["Duration"] = pd.to_numeric(df["Duration"], errors="coerce")

# Votes from "1,086" + log transform
df["Votes"] = df["Votes"].astype(str).str.replace(",", "", regex=False)
df["Votes"] = pd.to_numeric(df["Votes"], errors="coerce")
df["Votes"] = np.log1p(df["Votes"])

# -------------------------------------------------
# 3. Drop invalid rows
# -------------------------------------------------
df = df.dropna(subset=["Year", "Duration", "Votes", TARGET_COL])

# -------------------------------------------------
# 4. Encode Genre (multi-hot)
# -------------------------------------------------
df["Genre"] = df["Genre"].astype(str).str.split(", ")

mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(df["Genre"])

genre_df = pd.DataFrame(
    genre_encoded,
    columns=["genre_" + g for g in mlb.classes_],
    index=df.index
)

df = pd.concat([df.drop(columns=["Genre"]), genre_df], axis=1)

# -------------------------------------------------
# 5. Split X and y
# -------------------------------------------------
y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])

# Drop high-cardinality junk
DROP_COLS = ["Name", "Actor 1", "Actor 2", "Actor 3"]
X = X.drop(columns=[c for c in DROP_COLS if c in X.columns])

# -------------------------------------------------
# 6. Feature detection
# -------------------------------------------------
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# -------------------------------------------------
# 7. Preprocessing pipeline
# -------------------------------------------------
transformers = []

if numeric_features:
    transformers.append(("num", "passthrough", numeric_features))

if categorical_features:
    transformers.append(
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    )

preprocessor = ColumnTransformer(transformers)

# -------------------------------------------------
# 8. Model
# -------------------------------------------------
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=1
)

pipeline = make_pipeline(preprocessor, model)

# -------------------------------------------------
# 9. Train/test split (for evaluation)
# -------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# -------------------------------------------------
# 10. Train
# -------------------------------------------------
pipeline.fit(X_train, y_train)

# -------------------------------------------------
# 11. Evaluation
# -------------------------------------------------
y_pred = pipeline.predict(X_test)

print("\nEvaluation:")
print("MAE :", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2  :", r2_score(y_test, y_pred))

# -------------------------------------------------
# 12. Cross-validation
# -------------------------------------------------
cv_scores = cross_val_score(
    pipeline,
    X, y,
    cv=5,
    scoring="r2"
)

print("\nCross-validation:")
print("CV R2 mean:", cv_scores.mean())
print("CV R2 std :", cv_scores.std())

# -------------------------------------------------
# 13. Train FINAL model on full data
# -------------------------------------------------
pipeline.fit(X, y)

# -------------------------------------------------
# 14. Save model and metadata
# -------------------------------------------------
joblib.dump(pipeline, "imdb_rating_model.pkl")

joblib.dump({
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "genre_classes": mlb.classes_.tolist(),
    "r2_estimate": float(cv_scores.mean())
}, "imdb_model_metadata.pkl")

print("\nModel saved successfully.")
    

Numeric features: ['Year', 'Duration', 'Votes']
Categorical features: ['Director']

Evaluation:
MAE : 0.8812988898377453
RMSE: 1.2043214812271574
R2  : 0.2204367616984706

Cross-validation:
CV R2 mean: 0.26430179333667475
CV R2 std : 0.028992083510151853

Model saved successfully.
